In [1]:
import matplotlib as mpl
import matplotlib.font_manager as mplf

import asdf
from asdf_settings.rapidlooks import TITILLIUM
from rapid.algebra import make_classifier_look
from rapid.helpers import get_zcam_bandset
from scipy.ndimage import gaussian_filter

mpl.rcParams["image.interpolation"] = None
mpl.rcParams["figure.dpi"] = 250

mpl.rcParams['image.interpolation'] = None
mpl.rcParams['figure.dpi'] = 250

%matplotlib qt

In [2]:
image_path = "/home/michael/Desktop/zcam_data/products/0130/iof/"
roi_path = None
# observation is a ZcamBandSet object -- a subclass of our multispectral
# data multitool/abstraction with special behavior for ZCAM.
observation = get_zcam_bandset(image_path, roi_path)

In [3]:
# the observation object knows lots of useful things about ZCAM in general 
# and these images in particular.
observation.wavelength('R1'), observation.summary['SOLAR_ELEVATION']

([800.0], 81.3009)

the syntax for defining spectral parameters is fairly straightforward. notes: 
* for band depth, bands are ordered (left, right, center). 
* allowable values for 'op' are: band_depth, ratio, slope, band_avg (average reflectance across all bands between first and second band inclusive), band_min (wavelength of minimum-valued band between first and second band inclusive), band_max (wavelength of maximum-valued band between first and second band inclusive).

In [4]:
parameter_definitions = {
    "BD866": {"op": "band_depth", "bands": ("R1", "R6", "R2")},
    "BD910": {"op": "band_depth", "bands": ("R1", "R5", "R3")},
    "BD910_wide": {"op": "band_depth", "bands": ("R1", "R6", "R3")},
    "BD939": {"op": "band_depth", "bands": ("R1", "R6", "R4")},
    "BD978": {"op": "band_depth", "bands": ("R1", "R6", "R5")},
    "R800_1022": {"op": "ratio", "bands": ("R1", "R6")},
    "R631_800": {"op": "ratio", "bands": ("R0R", "R1")},
    "R800_631": {"op": "ratio", "bands": ("R1", "R0R")},
}

syntax for these class definitions is also pretty straightforward. notes:  
* you can't use chained relational operators -- e.g., '0.08 <= BD866 <= 0.2'
* '&' means logical and
* we can eventually do non-relational algebras too if you've got interest, but the plotter functions i've currently defined will only be happy if they get boolean arrays, so make sure that any expression you write ends up evaluating to True/False
* class definitions flow top-to-bottom exclusive (like IDL CASE, as opposed to IDL SWITCH) -- a pixel is colored in the consolidated classification map according to the first class it falls into. 

* any [standard CSS color](https://www.w3.org/wiki/CSS/Properties/color/keywords) should be a legal value for 'color'.

In [5]:
spectral_classes = {
    "hematite": {
        "definition": "((0.08 <= BD866) & (BD866 <= 0.2))"
        "& ((1.2 <= R800_631) & (R800_631 <= 2.0))"
        "& (BD866 > BD910)",
        "color": "firebrick",
    },
    "red slope": {
        "definition": "((0.6 <= BD866) & (BD866 <= 0.9))"
        "& ((0.4 <= R800_631) & (R800_631 <= 0.8))",
        "color": "coral",
    },
    "blue slope": {
        "definition": "((1.1 <= R800_1022) & (R800_1022 <= 1.4))"
        "& ((1.0 <= R631_800) & (R631_800 <= 1.6))",
        "color": "blue",
    },
    ">900 nm": {
        "definition": "((0.03 <= BD939) & (BD939 <=0.2))"
        "& ((0.05 <= BD978) & (BD978 <= 0.2))"
        "& (BD939 > BD910)",
        "color": "darkgray",
    },
    "900 nm": {
        "definition": "((0.03 <= BD910) & (BD910 <= 0.2))"
        "& ((0.03 <= BD939) & (BD939 <= 0.2))"
        "& (BD910_wide > BD866)",
        "color": "darkorchid",
    },
}

In [6]:
# font size for legends
font_size = 12

# a recommended prefilter. consider playing with the sigma parameter.
prefilter = {"function": gaussian_filter, "params": {"sigma": 1}}
# a different filter that's sensitive to edges.
# prefilter = {"function": make_bilateralfilter(15, 3, 7)}

# this is 'experimental' functionality, so it has its very own assembler.
class_looks = make_classifier_look(
    observation,
    spectral_classes,
    parameter_definitions,
    # if you don't want to use a filter, just set this to None
    prefilter=prefilter,
    fontproperties=mplf.FontProperties(fname=TITILLIUM, size=font_size),
    # if True, renders maps for each class along with the consolidated map
    plot_all=True,
    # if you set this to False, you will just get arrays back, not images
    plot=True,
)

class_plots = class_looks.execute()

In [7]:
# you can also render the standard asdf rapidlooks from this notebook, 
# although font sizes etc. may not be beautifully perfect
# observation.make_look_set(
#     [rapidlooks['slope R1_R6'], rapidlooks['enhanced color L0R_L0G_L0B']]
# )

In [8]:
# save a look -- again, maybe mess with font sizes, WIP, etc.
# save_plainly(class_plots[0], '.', 'spectral_class_map.png', dpi=150)

In [9]:
# run to close all matplotlib figures
# plt.close('all')